# Семантическое обогащение данных о местах

В работе используется выборка московских POI из OpenStreetMap. Из исходных тегов строятся признаки для рекомендаций, после чего сравнивается поиск по исходным и обогащённым данным.

## 1. Окружение

Здесь устанавливаются библиотеки, фиксируется seed и задаются пути к сохранённым данным.

In [ ]:
%pip install -q "pandas==3.1.0" "numpy==2.5.4" "scikit-learn==1.9.1" "matplotlib==3.11.2" "seaborn==0.13.2" "requests==2.32.5"

In [ ]:
import json
import platform
import random
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RAW_PATH = Path("places_raw.csv")
ENRICHED_PATH = Path("places_enriched.csv")
RESULTS_PATH = Path("search_results_comparison.csv")

print("Python:", platform.python_version())
for package in ["pandas", "numpy", "scikit-learn"]:
    print(f"{package}: {version(package)}")

## 2. Получение и отбор мест

Снимок OpenStreetMap загружается из `places_raw.csv`, поэтому обычный запуск не обращается к API. Повторный сбор включается отдельным параметром и нужен только для обновления данных.


In [ ]:
import time
from datetime import date

import requests

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "vk-internship-poi-project/1.0 (educational project)"
VIEWBOX = "37.2,56.0,38.0,55.5"
CATEGORY_QUERIES = {
    "cafe": "кафе, Москва",
    "restaurant": "ресторан, Москва",
    "museum": "музей, Москва",
    "park": "парк, Москва",
    "theatre": "театр, Москва",
    "library": "библиотека, Москва",
    "cinema": "кинотеатр, Москва",
}

def collect_nominatim_snapshot(queries=CATEGORY_QUERIES, limit=10, delay=1.1):
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    rows = []
    for category, query in queries.items():
        response = session.get(NOMINATIM_URL, params={
            "q": query,
            "format": "jsonv2",
            "addressdetails": 1,
            "extratags": 1,
            "namedetails": 1,
            "bounded": 1,
            "viewbox": VIEWBOX,
            "limit": limit,
        }, timeout=30)
        response.raise_for_status()
        for item in response.json():
            address = item.get("address", {})
            tags = item.get("extratags", {})
            osm_type, osm_id = item["osm_type"], item["osm_id"]
            rows.append({
                "poi_id": f"{osm_type}/{osm_id}",
                "name": item.get("namedetails", {}).get("name") or item.get("name") or item["display_name"].split(",")[0],
                "category": category,
                "city": address.get("city") or address.get("town") or address.get("municipality", ""),
                "district": address.get("city_district") or address.get("borough") or address.get("suburb", ""),
                "address": item.get("display_name", ""),
                "latitude": float(item["lat"]),
                "longitude": float(item["lon"]),
                "opening_hours": tags.get("opening_hours", ""),
                "cuisine": tags.get("cuisine", ""),
                "wheelchair": tags.get("wheelchair", ""),
                "outdoor_seating": tags.get("outdoor_seating", ""),
                "internet_access": tags.get("internet_access", ""),
                "website": tags.get("website") or tags.get("contact:website", ""),
                "wikidata": tags.get("wikidata", ""),
                "wikipedia": tags.get("wikipedia", ""),
                "osm_url": f"https://www.openstreetmap.org/{osm_type}/{osm_id}",
                "source": "OpenStreetMap via Nominatim",
                "source_license": "ODbL 1.0",
                "retrieved_at": date.today().isoformat(),
                "raw_tags": json.dumps(tags, ensure_ascii=False, sort_keys=True),
            })
        time.sleep(delay)
    return pd.DataFrame(rows).drop_duplicates("poi_id").reset_index(drop=True)

def load_snapshot(path, refresh=False):
    if refresh:
        snapshot = collect_nominatim_snapshot()
        snapshot.to_csv(path, index=False)
        return snapshot
    if path.exists():
        return pd.read_csv(path).fillna("")
    raise FileNotFoundError(f"Missing snapshot: {path.resolve()}")

raw_snapshot = load_snapshot(RAW_PATH)
required = {
    "poi_id", "name", "category", "city", "address", "latitude", "longitude",
    "osm_url", "source", "source_license", "retrieved_at", "raw_tags",
}
if not required <= set(raw_snapshot.columns):
    raise ValueError("The POI snapshot has an unexpected schema")

inside_moscow = raw_snapshot["address"].str.contains("Москва", case=False, na=False) & ~raw_snapshot["address"].str.contains("Московская область", case=False, na=False)
park_names = raw_snapshot["name"].str.contains(r"парк|сад|сквер|заказник|холм|бульвар|набережн|лес", case=False, na=False)
is_place = raw_snapshot["category"].ne("park") | park_names
places_raw = raw_snapshot.loc[inside_moscow & is_place].drop_duplicates("poi_id").reset_index(drop=True)
rejected = raw_snapshot.loc[~(inside_moscow & is_place), ["name", "category", "address"]]

if not 50 <= len(places_raw) <= 100 or not places_raw["poi_id"].is_unique:
    raise ValueError("The filtered snapshot must contain 50–100 unique places")

category_counts = places_raw["category"].value_counts().sort_index()
print(f"Raw rows: {len(raw_snapshot)}")
print(f"Selected places: {len(places_raw)}")
print(f"Rejected rows: {len(rejected)}")
display(category_counts.to_frame("places"))
rejected


## 3. Семантические признаки

Для каждой категории задаётся один компактный профиль: тип опыта, атмосфера, сценарий посещения, время и русские поисковые термины. Доступные OSM-теги дополняют профиль сведениями о Wi-Fi, террасе и доступности.


In [ ]:
PROFILE_COLUMNS = [
    "category", "label", "experience_types", "atmospheres",
    "visit_contexts", "visit_times", "search_terms",
]
PROFILE_ROWS = [
    ("museum", "музей", "culture|education", "calm|indoor", "family|weekend", "day|weekend", "музей выставка культура образование спокойно семья выходные"),
    ("cafe", "кафе", "food|work", "cozy|indoor", "work|friends", "morning|day", "кафе кофе еда работа уютное ноутбук друзья утро"),
    ("restaurant", "ресторан", "food|social", "cozy|social", "friends|date", "day|evening", "ресторан ужин еда уютное романтическое свидание друзья вечер"),
    ("park", "парк", "walk|nature", "outdoor|calm|quiet", "family|walk", "day|weekend", "парк прогулка природа на улице спокойно тихое семья выходные отдых"),
    ("theatre", "театр", "culture|entertainment", "cultural|indoor", "date|evening", "evening", "театр спектакль культура развлечение романтическое свидание вечер"),
    ("library", "библиотека", "education|work", "quiet|indoor", "work|alone", "day", "библиотека чтение образование работа тихое спокойно одному отдых"),
    ("cinema", "кинотеатр", "culture|entertainment", "entertainment|indoor", "friends|date", "day|evening", "кино кинотеатр фильм развлечение романтическое друзья свидание вечер"),
]
category_profiles = pd.DataFrame(PROFILE_ROWS, columns=PROFILE_COLUMNS).set_index("category")
for column in ["experience_types", "atmospheres", "visit_contexts", "visit_times"]:
    category_profiles[column] = category_profiles[column].str.split("|")

TAG_SIGNALS = [
    ("outdoor_seating", {"yes"}, "atmospheres", "outdoor", "на улице терраса"),
    ("internet_access", {"yes", "wlan", "wifi"}, "visit_contexts", "work", "работа wi-fi ноутбук"),
    ("wheelchair", {"yes"}, "visit_contexts", "accessible", "доступность коляска"),
]

def extract_features(row):
    profile = category_profiles.loc[row.category]
    features = {
        field: list(profile[field])
        for field in ["experience_types", "atmospheres", "visit_contexts", "visit_times"]
    }
    evidence = [f"category={row.category}"]
    search_terms = [profile["search_terms"]]
    for source, accepted, target, value, terms in TAG_SIGNALS:
        if str(getattr(row, source)).lower() in accepted:
            features[target].append(value)
            evidence.append(f"{source}={getattr(row, source)}")
            search_terms.append(terms)
    completeness_fields = [row.opening_hours, row.cuisine, row.wheelchair, row.website, row.wikidata, row.wikipedia]
    return {
        **{field: list(dict.fromkeys(values)) for field, values in features.items()},
        "budget": "unknown",
        "metadata_completeness": np.mean([bool(value) for value in completeness_fields]),
        "feature_evidence": evidence,
        "available_tags": sorted(json.loads(row.raw_tags)) if row.raw_tags else [],
        "semantic_search_terms": " ".join(search_terms),
    }

places = pd.concat([
    places_raw.reset_index(drop=True),
    pd.DataFrame(extract_features(row) for row in places_raw.itertuples(index=False)),
], axis=1)

def make_description(row):
    district = f" в районе {row.district}" if row.district else " в Москве"
    details = [value for value in [
        f"кухня: {row.cuisine}" if row.cuisine else "",
        f"часы работы: {row.opening_hours}" if row.opening_hours else "",
    ] if value]
    suffix = f" Дополнительно: {'; '.join(details)}." if details else ""
    return f"{row['name']} — {category_profiles.loc[row.category, 'label']}{district}. {row.semantic_search_terms}.{suffix}"

places["description"] = places.apply(make_description, axis=1)
enriched_export = places.copy()
for column in ["experience_types", "atmospheres", "visit_contexts", "visit_times", "feature_evidence", "available_tags"]:
    enriched_export[column] = enriched_export[column].map(lambda values: json.dumps(values, ensure_ascii=False))
enriched_export.to_csv(ENRICHED_PATH, index=False)
display(places.head()[["name", "category", "description", "feature_evidence"]])


## 4. Проверка и EDA

Считается полнота дополнительных OSM-полей. Графики показывают состав выборки и частотность многозначных семантических признаков.

In [ ]:
coverage = places[[
    "opening_hours", "cuisine", "wheelchair", "website", "wikidata", "wikipedia"
]].ne("").mean().rename("coverage")
coverage.to_frame().round(3)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

category_counts = places["category"].value_counts()
sns.barplot(x=category_counts.values, y=category_counts.index, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Categories")
axes[0, 0].set_xlabel("Places")
axes[0, 0].set_ylabel("")

experience_counts = places.explode("experience_types")["experience_types"].value_counts()
sns.barplot(x=experience_counts.values, y=experience_counts.index, ax=axes[0, 1], color="seagreen")
axes[0, 1].set_title("Experience types")
axes[0, 1].set_xlabel("Places")
axes[0, 1].set_ylabel("")

atmosphere_counts = places.explode("atmospheres")["atmospheres"].value_counts()
sns.barplot(x=atmosphere_counts.values, y=atmosphere_counts.index, ax=axes[1, 0], color="darkorange")
axes[1, 0].set_title("Atmospheres")
axes[1, 0].set_xlabel("Places")
axes[1, 0].set_ylabel("")

context_counts = places.explode("visit_contexts")["visit_contexts"].value_counts()
sns.barplot(x=context_counts.values, y=context_counts.index, ax=axes[1, 1], color="mediumpurple")
axes[1, 1].set_title("Visit contexts")
axes[1, 1].set_xlabel("Places")
axes[1, 1].set_ylabel("")

plt.tight_layout()
plt.show()

## 5. Поиск

Сравниваются два TF-IDF индекса. Первый использует только метаданные места, второй добавляет семантический профиль. Оба индекса объединяют словные и символьные признаки, поэтому отдельный набор ручных правил для каждого запроса не нужен.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import FeatureUnion

SEARCH_CUTOFFS = (5, 10, 15)

places["metadata_text"] = places.apply(
    lambda row: " ".join(str(value) for value in [
        row["name"], category_profiles.loc[row.category, "label"],
        row.cuisine, row.district, row.address,
    ] if value),
    axis=1,
)
places["enriched_text"] = places["metadata_text"] + " " + places["semantic_search_terms"]

search_spaces = {}
for method, column in {"metadata": "metadata_text", "enriched": "enriched_text"}.items():
    vectorizer = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)),
    ])
    search_spaces[method] = (vectorizer, vectorizer.fit_transform(places[column]))

def search_places(query, method, k=max(SEARCH_CUTOFFS)):
    vectorizer, matrix = search_spaces[method]
    scores = cosine_similarity(vectorizer.transform([query]), matrix).ravel()
    order = np.argsort(-scores)[:k]
    result = places.iloc[order][["poi_id", "name", "category", "description", "osm_url"]].copy()
    result["score"] = scores[order]
    return result

QUERY_SPECS = [
    {"query_id": "work_cafe", "query": "Уютное кафе для работы", "categories": {"cafe"}, "semantics": {"work", "cozy"}},
    {"query_id": "date", "query": "Романтическое место для свидания", "categories": {"restaurant", "theatre", "cinema"}, "semantics": {"date"}},
    {"query_id": "friends_dinner", "query": "Недорогое место для ужина с друзьями", "categories": {"restaurant", "cafe"}, "semantics": {"friends"}},
    {"query_id": "family_walk", "query": "Интересное место для прогулки с семьёй", "categories": {"park", "museum"}, "semantics": {"family", "walk"}},
    {"query_id": "quiet_rest", "query": "Тихое место для отдыха", "categories": {"park", "library"}, "semantics": {"quiet"}},
    {"query_id": "weekend_museum", "query": "Музей для посещения в выходные", "categories": {"museum"}, "semantics": {"weekend"}},
]

def reference_relevance(row, spec):
    semantics = set(row.experience_types) | set(row.atmospheres) | set(row.visit_contexts) | set(row.visit_times)
    return int(row.category in spec["categories"] and spec["semantics"].issubset(semantics))

result_rows = []
for spec in QUERY_SPECS:
    total_relevant = sum(reference_relevance(row, spec) for row in places.itertuples(index=False))
    for method in search_spaces:
        result = search_places(spec["query"], method)
        for rank, row in enumerate(result.itertuples(index=False), start=1):
            source_row = places.loc[places["poi_id"] == row.poi_id].iloc[0]
            result_rows.append({
                "query_id": spec["query_id"],
                "query": spec["query"],
                "method": method,
                "rank": rank,
                "poi_id": row.poi_id,
                "name": row.name,
                "category": row.category,
                "score": row.score,
                "reference_relevance": reference_relevance(source_row, spec),
                "total_relevant_in_corpus": total_relevant,
                "osm_url": row.osm_url,
            })

search_results = pd.DataFrame(result_rows)
relevant_at_max = search_results.groupby(["query_id", "method"])["reference_relevance"].transform("sum").astype(int)
search_results["analysis"] = relevant_at_max.astype(str) + f"/{max(SEARCH_CUTOFFS)} relevant results"
search_results.to_csv(RESULTS_PATH, index=False)
search_results.head(10)


## 6. Sanity-check поиска

На глубинах 5, 10 и 15 считаются Precision, Recall, MAP, nDCG, MRR и HitRate. Это проверка согласованности поиска с разметкой, а не независимая оценка пользователями.

In [ ]:
def ranking_metrics(values, total_relevant, k):
    hits = np.asarray(values[:k], dtype=float)
    ranks = np.arange(1, k + 1)
    cumulative = np.cumsum(hits)
    discounts = 1 / np.log2(ranks + 1)
    ideal = np.zeros(k)
    ideal[:min(total_relevant, k)] = 1
    return {
        "precision": float(hits.mean()),
        "recall": float(hits.sum() / total_relevant) if total_relevant else 0.0,
        "map": float(np.sum((cumulative / ranks) * hits) / min(total_relevant, k)) if total_relevant else 0.0,
        "ndcg": float(np.sum(hits * discounts) / np.sum(ideal * discounts)) if total_relevant else 0.0,
        "mrr": float(1 / ranks[np.flatnonzero(hits)[0]]) if hits.any() else 0.0,
        "hit_rate": float(hits.any()),
    }

metric_rows = []
for (query_id, method), group in search_results.groupby(["query_id", "method"]):
    ordered = group.sort_values("rank")
    total_relevant = int(ordered["total_relevant_in_corpus"].iloc[0])
    for k in SEARCH_CUTOFFS:
        metric_rows.append({
            "query_id": query_id,
            "method": method,
            "k": k,
            "total_relevant_in_corpus": total_relevant,
            **ranking_metrics(ordered["reference_relevance"].to_numpy(), total_relevant, k),
        })

search_metrics = pd.DataFrame(metric_rows)
metric_columns = ["precision", "recall", "map", "ndcg", "mrr", "hit_rate"]
summary_metrics = search_metrics.groupby(["method", "k"])[metric_columns].mean()
display(summary_metrics.round(3))
display(search_metrics.pivot_table(index=["query_id", "k"], columns="method", values="ndcg").round(3))


In [ ]:
for spec in QUERY_SPECS:
    print(spec["query"])
    display(search_results[
        (search_results["query_id"] == spec["query_id"])
        & (search_results["method"] == "enriched")
        & (search_results["rank"] <= 5)
    ][["rank", "name", "category", "score", "reference_relevance"]])


## 7. Выводы

Индекс с семантическими профилями лучше обрабатывает запросы для работы, свидания или семейной прогулки. В сравнительный CSV добавлен краткий разбор каждой пары «запрос — метод».

Sanity-check не заменяет пользовательскую оценку: релевантность задаётся теми же типами признаков, которые используются при обогащении. OSM-теги разрежены, а цены, рейтинги и отзывы отсутствуют. Поэтому бюджет остаётся `unknown`, а запрос «недорогое место» оценивается только по ужину и компании друзей.

In [ ]:
REFERENCE_K = 10
reference_metrics = summary_metrics.xs(REFERENCE_K, level="k")
best_method = reference_metrics["ndcg"].idxmax()
print(f"Places: {len(places)}")
print(f"Categories: {places['category'].nunique()}")
print(f"Search queries: {len(QUERY_SPECS)}")
print(f"Best method by nDCG@{REFERENCE_K}: {best_method}")
display(summary_metrics.round(3))
